# OASIS Data

---

### package imports and basic functions

---

In [2]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl


In [3]:
from spectranorm import snm

In [4]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Extracting data

---

In [5]:
dataset = 'OASIS'

In [4]:
data_info_df = pd.read_csv("/mnt/nas/CSC7/Yeolab/Data/OASIS/users_data/Sina/OASIS.csv")
data_info_df.shape


(9079, 6)

In [ ]:
data_info_df.head(10)

In [6]:
data_info_df[pd.notna(data_info_df['Scan_path'])][['DX']].value_counts(dropna=False)

DX 
CN     1701
NaN     570
DEM     233
AD       82
MCI      54
Name: count, dtype: int64

In [8]:
data_info_df[pd.notna(data_info_df['Scan_path'])][['Scanner_info']].value_counts(dropna=False)

Scanner_info             
Siemens/TrioTim/3.0          1427
Siemens/Biograph_mMR/3.0      837
Siemens/MAGNETOM_Vida/3.0     333
Siemens/Sonata/1.494           22
Siemens/Sonata/1.5             18
Siemens/Avanto/1.5              2
Siemens/Prisma_fit/3.0          1
Name: count, dtype: int64

In [ ]:
valid_subjects_dict = {}

for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["Scan_path"]):
        key = row["Scan_path"].split("/")[-1]
        valid_subjects_dict[key] = {
            "unique_id": key,
            "participant_id": "_".join(key.split("_")[0:1]),
            "session_id": key.split("_")[-1],
            "site": str(row["Scanner_info"]),
            "sex": row["Sex"],
            "age": row["Age"],
            "diagnosis": row["DX"],
            "scan_path": row["Scan_path"],
        }

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [25]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def directory_is_valid(path, items = items):
    return len([f for f in items if (path / f).exists()]) == len(items)

In [ ]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]

    freesurfer_directory = valid_subjects_dict[subject]["scan_path"]
    
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/{dataset}/{sub_dir}/{subject}.thickness.fslr.npy"

    if (directory_is_valid(Path(freesurfer_directory) / "surf")) and (not Path(thickness_fslr_output).exists()):    
        # Compute fslr thickness
        transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
        np.save(
            ensure_dir(thickness_fslr_output),
            transformed_fslr_thickness.astype(np.float32)
        )


  0%|          | 0/2640 [00:00<?, ?it/s]

In [ ]:
%%time
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]
    valid_subjects_dict[subject]["subject_index"] = idx
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/{dataset}/{sub_dir}/{subject}.thickness.fslr.npy"
    if Path(thickness_fslr_output).exists():
        valid_subjects_dict[subject]["thickness"] = np.load(
            thickness_fslr_output,
        ).mean()
    else:
        valid_subjects_dict[subject]["thickness"] = np.nan

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [32]:
eno_items = [
    "lh.orig.nofix", "rh.orig.nofix",
]

# Compute Euler Number
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]
    freesurfer_directory = valid_subjects_dict[subject]["scan_path"]
    if ("euler_no" not in valid_subjects_dict[subject]) or np.isnan(valid_subjects_dict[subject]["euler_no"]):
        if (directory_is_valid(Path(freesurfer_directory) / "surf", items=eno_items)):    
            # Compute euler number
            valid_subjects_dict[subject]["euler_no"] = snm.utils.nitools.compute_total_euler_number(
                Path(freesurfer_directory)
            )
        else:
            valid_subjects_dict[subject]["euler_no"] = np.nan


  0%|          | 0/2640 [00:00<?, ?it/s]

In [ ]:
len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [33]:
len([x for x in valid_subjects_dict if valid_subjects_dict[x]['euler_no'] == np.nan])

0

In [34]:
# Validity checks
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    # if "validity_check" not in valid_subjects_dict[subject]:
    valid_subjects_dict[subject]["validity_check"] = (
        (valid_subjects_dict[subject]["diagnosis"] == 'CN')  # Exclude those with a diagnosis
        and
        not np.isnan(valid_subjects_dict[subject]["thickness"])  # Exclude those missing thickness data
        and
        not np.isnan(valid_subjects_dict[subject]["euler_no"])  # Exclude those missing Euler number
    )


  0%|          | 0/2640 [00:00<?, ?it/s]

In [35]:
import joblib

joblib.dump(valid_subjects_dict, ensure_dir(f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"))


['/home/sina/storage/Normative_Modeling/data/datasets/OASIS/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict = joblib.load(
    f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"
)

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[key]["age"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'thickness': [valid_subjects_dict[key]["thickness"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'sex': [valid_subjects_dict[key]["sex"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'site': [valid_subjects_dict[key]["site"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[key]["participant_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'euler_no': [valid_subjects_dict[key]["euler_no"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[key]["unique_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_index': [valid_subjects_dict[key]["subject_index"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [38]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(919, 9)

# ✅ Finished!
